# Simple maps
* https://simplemaps.com/data/world-cities

In [0]:
%sql
SELECT * FROM csv.`/Volumes/workspace/default/databricks/spatial/worldcities.csv` --OPTIONS (header=>"true", inferSchema=>"true")

In [0]:
%sql
SELECT * FROM read_files(
  '/Volumes/workspace/default/databricks/spatial/worldcities.csv',
  format => 'csv',
  header => true,
  inferSchema => 'true'
)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW point_table AS
SELECT * FROM VALUES 
    (34.56, 137.2), --tokyo, jp
    (13.6, 100.23), -- bangkok, th
    (28.63, 76.3), -- delhi, in
    (28.64336,77.11756)
AS t(latitude, longitude);

In [0]:
%sql
WITH DistanceCalculations AS (
  SELECT 
    p.longitude,
    p.latitude,
    s.city AS site_column,
    s.country AS country,
    -- Calculates spherical distance in meters between the two points
    ST_DistanceSphere(
      ST_Point(p.longitude, p.latitude),
      ST_Point(s.lng, s.lat)
    ) AS distance_meters,
    ROW_NUMBER() OVER (
      PARTITION BY p.longitude, p.latitude 
      ORDER BY ST_DistanceSphere(
        ST_Point(p.longitude, p.latitude),
        ST_Point(s.lng, s.lat)
      ) ASC
    ) AS rank
  FROM point_table p
  CROSS JOIN (
  SELECT * FROM read_files('/Volumes/workspace/default/databricks/spatial/worldcities.csv',
    format => 'csv',
    header => true,
    inferSchema => 'true'
    )) s 
  
)
SELECT 
  longitude,
  latitude,
  site_column,
  country,
  distance_meters
FROM DistanceCalculations
WHERE rank = 1;
     